# Spotify Track Dataset - EDA

Objective: inspect the Spotify track dataset, understand data quality, explore important audio features, and prepare feature candidates for recommendation and prediction models.

## 1. Setup

Import libraries and configure display options.

In [ ]:
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 120)
sns.set_theme(style="whitegrid")

## 2. Load Dataset

In [ ]:
PROJECT_ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "data").exists())
DATA_PATH = PROJECT_ROOT / "data" / "raw" / "dataset.csv"

df = pd.read_csv(DATA_PATH)
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]:,}")
df.head()

## 3. Basic Structure

In [ ]:
df.info()

In [ ]:
df.describe(include="all").T

## 4. Missing Values And Duplicates

In [ ]:
missing = df.isna().sum().sort_values(ascending=False)
missing[missing > 0]

In [ ]:
duplicate_track_ids = df["track_id"].duplicated().sum()
print(f"Duplicate track_id rows: {duplicate_track_ids:,}")

df[df.duplicated("track_id", keep=False)].sort_values("track_id").head(20)

## 5. Initial Cleaning For EDA

This creates an EDA-safe copy. The raw dataframe stays unchanged.

In [ ]:
eda = df.copy()
eda = eda.drop(columns=["Unnamed: 0"], errors="ignore")
eda = eda.dropna(subset=["track_name", "artists"])
eda["album_name"] = eda["album_name"].fillna("Unknown")
eda["duration_min"] = eda["duration_ms"] / 60000
eda["explicit_int"] = eda["explicit"].astype(int)

text_cols = ["artists", "album_name", "track_name", "track_genre"]
for col in text_cols:
    eda[col] = eda[col].astype(str).str.strip()

print(eda.shape)
eda.head()

## 6. Validate Numeric Ranges

In [ ]:
range_checks = {
    "popularity_outside_0_100": eda[(eda["popularity"] < 0) | (eda["popularity"] > 100)].shape[0],
    "duration_ms_lte_0": eda[eda["duration_ms"] <= 0].shape[0],
    "tempo_lte_0": eda[eda["tempo"] <= 0].shape[0],
}
range_checks

In [ ]:
audio_cols = [
    "danceability", "energy", "loudness", "speechiness", "acousticness",
    "instrumentalness", "liveness", "valence", "tempo", "duration_min"
]

eda[audio_cols + ["popularity"]].describe().T

## 7. Genre And Popularity Overview

In [ ]:
print(f"Number of genres: {eda['track_genre'].nunique()}")
eda["track_genre"].value_counts().head(20)

In [ ]:
plt.figure(figsize=(10, 5))
sns.histplot(eda["popularity"], bins=30, kde=True)
plt.title("Popularity Distribution")
plt.xlabel("Popularity")
plt.ylabel("Track Count")
plt.show()

In [ ]:
top_genres = eda.groupby("track_genre")["popularity"].mean().sort_values(ascending=False).head(15)

plt.figure(figsize=(10, 6))
sns.barplot(x=top_genres.values, y=top_genres.index)
plt.title("Top Genres By Average Popularity")
plt.xlabel("Average Popularity")
plt.ylabel("Genre")
plt.show()

## 8. Audio Feature Distributions

In [ ]:
eda[audio_cols].hist(figsize=(16, 12), bins=30)
plt.suptitle("Audio Feature Distributions", y=1.02)
plt.tight_layout()
plt.show()

## 9. Which Song Features Relate To Popularity?

This section is the main analytical question of the notebook. It compares:

- **Pearson correlation** for linear relationships.
- **Spearman correlation** for monotonic relationships that may not be linear.
- **Within-genre correlations** to reduce the chance that a relationship is only reflecting differences between genres.

Before calculating these relationships, the analysis keeps one row per track ID so repeated catalog entries do not receive extra weight. These are associations, not causal effects. Popularity also depends on artist fame, promotion, playlist placement, timing, and social trends that are not available in this dataset.

In [ ]:
import sys

src_path = str(PROJECT_ROOT / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

from spotify_eda_analysis import (
    categorical_popularity_effects,
    plot_popularity_correlations,
    plot_top_feature_trends,
    prepare_popularity_analysis_data,
    popularity_correlations,
)

In [ ]:
popularity_data = prepare_popularity_analysis_data(eda)
print(f"Popularity analysis rows after track-ID deduplication: {len(popularity_data):,}")

feature_correlations = popularity_correlations(popularity_data)
feature_correlations[[
    "feature",
    "pearson",
    "spearman",
    "within_genre_pearson",
    "within_genre_spearman",
]].round(3)

In [ ]:
plot_popularity_correlations(feature_correlations)
plt.show()

### How To Read The Correlations

Rank features using the **absolute within-genre Spearman correlation** first. This is the most conservative descriptive view here because it allows non-linear monotonic patterns and compares tracks within their genre context. Then compare it with the overall correlation:

- Similar overall and within-genre values suggest a relationship that is relatively stable across genre context.
- A much smaller within-genre value suggests that genre mix explains part of the overall pattern.
- Values near zero do not prove that a feature is irrelevant; tree-based models may still use thresholds and interactions.

In [ ]:
plot_top_feature_trends(popularity_data, feature_correlations, top_n=6)
plt.show()

### Discrete Composition Features

For key, mode, time signature, and explicit status, ordinary numeric correlation is not an appropriate primary measure. Eta-squared reports how much of the observed popularity variance is associated with differences between each feature's categories.

In [ ]:
categorical_effects = categorical_popularity_effects(popularity_data)
categorical_effects.round(4)

## 10. Candidate Features For Recommendation And Modeling

Correlation is useful for interpretation, but it is not a feature-selection rule by itself. The recommender keeps perceptual audio features because they define similarity. The popularity model keeps composition features and genre so it can learn thresholds and interactions while excluding direct artist-fame identifiers.

In [ ]:
candidate_features = [
    "track_genre", "danceability", "energy", "loudness", "speechiness",
    "acousticness", "instrumentalness", "liveness", "valence",
    "tempo", "duration_min", "explicit_int", "popularity"
]

eda[candidate_features].head()

## 11. Save Cleaned EDA Dataset

In [ ]:
OUTPUT_PATH = PROJECT_ROOT / "data" / "processed" / "spotify_cleaned_eda.csv"
eda.to_csv(OUTPUT_PATH, index=False)
print(f"Saved cleaned EDA dataset to: {OUTPUT_PATH}")

## EDA Takeaway

Use the within-genre results as the clearest descriptive evidence for which composition features move with popularity. Expect individual correlations and categorical effect sizes to be modest: popularity is a social and commercial outcome, and audio features capture only part of that process. The modeling notebook therefore compares non-linear models against a mean baseline and reports out-of-sample error rather than claiming that the selected features cause popularity.